# Обучение модели YOLOv8 для распознавания показаний счётчиков

## Цель
Переобучить предобученную модель **YOLOv8n** (nano) для детекции цифр (0–9) на изображениях счётчиков электроэнергии.

## Подход — Transfer Learning
- **Базовая модель**: YOLOv8n, предобученная на COCO (80 классов, ~3.2M параметров)
- **Стратегия**: замораживаем backbone (слои 0–9, feature extractor), переобучаем только detection head
- **Датасет**: OCR Meter Reading (Roboflow, 5.5k изображений, 10 классов — цифры 0–9)
- **Экспорт**: TFLite для деплоя на Android

## Пайплайн распознавания
1. Камера → фото счётчика
2. YOLOv8n детектирует отдельные цифры (bounding boxes + class 0–9)
3. Сортировка bbox по x-координате (слева направо)
4. Конкатенация классов → показание счётчика (например: `0`, `2`, `4`, `7`, `3` → `02473`)

---
## 0. Подключение GPU

Перед запуском убедитесь, что в Colab выбран GPU:
**Runtime → Change runtime type → T4 GPU**

In [ ]:
# Проверяем доступность GPU
!nvidia-smi

---
## 1. Установка зависимостей

In [ ]:
!pip install ultralytics roboflow -q

In [ ]:
import ultralytics
ultralytics.checks()

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Загрузка датасета с Roboflow

Используем датасет **OCR Meter Reading** от UniLogic:
- 5 490 изображений счётчиков
- 10 классов: цифры `0`–`9`
- Лицензия: CC BY 4.0
- [Ссылка на датасет](https://universe.roboflow.com/unilogic/ocr-meter-reading)

### Получение API-ключа Roboflow
1. Зарегистрируйтесь на [roboflow.com](https://roboflow.com/) (бесплатно)
2. Перейдите в **Settings → API Key**
3. Скопируйте ключ и вставьте ниже

In [ ]:
from roboflow import Roboflow

# === ВСТАВЬТЕ ВАШ API-КЛЮЧ ROBOFLOW ===
ROBOFLOW_API_KEY = "YOUR_API_KEY"  # <-- замените на свой ключ

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("unilogic").project("ocr-meter-reading")
dataset = project.version(2).download("yolov8")

print(f"\nДатасет загружен в: {dataset.location}")

---
## 3. Исследование датасета (EDA)

In [ ]:
import os
import glob
import yaml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

# Читаем конфигурацию датасета
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("=== Конфигурация датасета ===")
print(f"Классы ({len(data_config['names'])}): {data_config['names']}")
print(f"Путь к train: {data_config.get('train', 'N/A')}")
print(f"Путь к val:   {data_config.get('val', 'N/A')}")
print(f"Путь к test:  {data_config.get('test', 'N/A')}")

# Считаем количество изображений в каждом сплите
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset.location, split, "images")
    if os.path.exists(img_dir):
        count = len(glob.glob(os.path.join(img_dir, "*")))
        print(f"\n{split}: {count} изображений")

In [ ]:
# Распределение классов в тренировочном наборе
class_names = data_config["names"]
label_dir = os.path.join(dataset.location, "train", "labels")
class_counts = Counter()
total_objects = 0

for label_file in glob.glob(os.path.join(label_dir, "*.txt")):
    with open(label_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                cls_id = int(parts[0])
                class_counts[cls_id] += 1
                total_objects += 1

print(f"Всего объектов (цифр) в train: {total_objects}\n")

# Гистограмма распределения классов
fig, ax = plt.subplots(figsize=(10, 5))
classes = sorted(class_counts.keys())
counts = [class_counts[c] for c in classes]
labels = [str(class_names[c]) if isinstance(class_names, list) else str(c) for c in classes]

bars = ax.bar(labels, counts, color="#4CAF50", edgecolor="black")
ax.set_xlabel("Класс (цифра)", fontsize=12)
ax.set_ylabel("Количество", fontsize=12)
ax.set_title("Распределение классов в тренировочном наборе", fontsize=14)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            str(count), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация примеров изображений с аннотациями
import cv2
from matplotlib import patches

COLORS = plt.cm.tab10(np.linspace(0, 1, 10))

def draw_annotations(img_path, label_path, class_names):
    """Отрисовка bounding box'ов на изображении."""
    img = Image.open(img_path)
    w, h = img.size

    fig, ax = plt.subplots(1, figsize=(8, 6))
    ax.imshow(img)

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

                # YOLO format → pixel coordinates
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                box_w = bw * w
                box_h = bh * h

                color = COLORS[cls_id % 10]
                rect = patches.Rectangle((x1, y1), box_w, box_h,
                                          linewidth=2, edgecolor=color, facecolor="none")
                ax.add_patch(rect)

                label = str(class_names[cls_id]) if isinstance(class_names, list) else str(cls_id)
                ax.text(x1, y1 - 4, label, color="white", fontsize=12, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8))

    ax.axis("off")
    return fig

# Показать 6 случайных примеров
train_imgs = glob.glob(os.path.join(dataset.location, "train", "images", "*"))
np.random.seed(42)
sample_imgs = np.random.choice(train_imgs, min(6, len(train_imgs)), replace=False)

for img_path in sample_imgs:
    basename = os.path.splitext(os.path.basename(img_path))[0]
    label_path = os.path.join(dataset.location, "train", "labels", basename + ".txt")
    fig = draw_annotations(img_path, label_path, class_names)
    plt.show()

---
## 4. Предобработка: аугментация в data.yaml

YOLOv8 применяет аугментации автоматически при обучении:
- Mosaic (комбинация 4 изображений)
- HSV augmentation (оттенок, насыщенность, яркость)
- Flip, Scale, Translate

Дополнительно добавим настройки для лучшей генерализации.

In [ ]:
# Обновляем data.yaml — указываем абсолютные пути
data_config["path"] = dataset.location
data_config["train"] = "train/images"
data_config["val"] = "valid/images"
data_config["test"] = "test/images"

with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("Обновлённый data.yaml:")
print(open(data_yaml_path).read())

---
## 5. Обучение модели

### Архитектура YOLOv8n

| Параметр | Значение |
|---|---|
| Backbone | CSPDarknet (слои 0–9) |
| Neck | PANet/FPN |
| Head | Decoupled detection head |
| Параметры | ~3.2M |
| Размер модели | ~6 MB |
| Input | 640×640 |

### Стратегия Transfer Learning
- `freeze=10` — замораживаем backbone (слои 0–9)
- Обучаем только neck + head на наших данных
- Это значительно быстрее полного обучения и предотвращает переобучение

In [ ]:
from ultralytics import YOLO

# Загружаем предобученную модель YOLOv8n (COCO weights)
model = YOLO("yolov8n.pt")

# Информация о модели
model.info()

In [ ]:
# === ОБУЧЕНИЕ С ЗАМОРОЗКОЙ BACKBONE ===
#
# freeze=10      — замораживаем backbone (слои 0–9), обучаем только голову
# epochs=50       — 50 эпох (можно увеличить до 100 для лучших результатов)
# imgsz=640       — размер входного изображения
# batch=16        — batch size (для T4 GPU в Colab)
# patience=15     — early stopping: остановка если val/mAP не растёт 15 эпох
# optimizer=AdamW — оптимизатор
# lr0=0.001       — начальный learning rate
# augment=True    — включить аугментации
# cos_lr=True     — cosine learning rate scheduler

results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    freeze=10,
    patience=15,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    device=0,
    project="meter_reading",
    name="yolov8n_transfer",
    verbose=True,
    plots=True
)

---
## 6. Анализ результатов обучения

In [ ]:
# Графики обучения (loss, mAP, precision, recall)
from IPython.display import display, Image as IPImage
import os

results_dir = "meter_reading/yolov8n_transfer"

# Кривые обучения
results_img = os.path.join(results_dir, "results.png")
if os.path.exists(results_img):
    print("=== Кривые обучения (Loss, mAP, Precision, Recall) ===")
    display(IPImage(filename=results_img, width=900))

In [ ]:
# Confusion Matrix
cm_img = os.path.join(results_dir, "confusion_matrix_normalized.png")
if os.path.exists(cm_img):
    print("=== Нормализованная матрица ошибок ===")
    display(IPImage(filename=cm_img, width=700))
else:
    cm_img = os.path.join(results_dir, "confusion_matrix.png")
    if os.path.exists(cm_img):
        print("=== Матрица ошибок ===")
        display(IPImage(filename=cm_img, width=700))

In [ ]:
# PR-кривая и F1-кривая
for name, title in [("PR_curve.png", "Precision-Recall кривая"),
                     ("F1_curve.png", "F1-Score кривая")]:
    path = os.path.join(results_dir, name)
    if os.path.exists(path):
        print(f"=== {title} ===")
        display(IPImage(filename=path, width=700))

---
## 7. Валидация на тестовом наборе

In [ ]:
# Загружаем лучшую модель
best_model_path = os.path.join(results_dir, "weights", "best.pt")
best_model = YOLO(best_model_path)

# Валидация
metrics = best_model.val(
    data=data_yaml_path,
    split="test",
    imgsz=640,
    batch=16,
    device=0
)

print("\n=== Метрики на тестовом наборе ===")
print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")

In [ ]:
# Per-class метрики
print("\n=== Метрики по классам ===")
print(f"{'Класс':<10} {'Precision':>10} {'Recall':>10} {'mAP@50':>10}")
print("-" * 45)

ap50_per_class = metrics.box.ap50
p_per_class = metrics.box.p
r_per_class = metrics.box.r

for i, name in enumerate(class_names if isinstance(class_names, list) else range(10)):
    if i < len(ap50_per_class):
        print(f"{str(name):<10} {p_per_class[i]:>10.4f} {r_per_class[i]:>10.4f} {ap50_per_class[i]:>10.4f}")

---
## 8. Тестирование на изображениях

In [ ]:
def predict_meter_reading(model, img_path, conf=0.25, imgsz=640):
    """
    Распознаёт показание счётчика:
    1. Детектирует цифры на изображении
    2. Сортирует по x-координате (слева направо)
    3. Возвращает строку показания
    """
    results = model.predict(img_path, conf=conf, imgsz=imgsz, verbose=False)
    result = results[0]

    if len(result.boxes) == 0:
        return "", result

    # Собираем детекции: (x_center, class_id, confidence)
    detections = []
    for box in result.boxes:
        x1 = box.xyxy[0][0].item()  # левая граница bbox
        cls_id = int(box.cls[0].item())
        conf_val = box.conf[0].item()
        detections.append((x1, cls_id, conf_val))

    # Сортируем по x-координате (слева направо)
    detections.sort(key=lambda d: d[0])

    # Формируем показание
    reading = "".join(str(d[1]) for d in detections)

    return reading, result


def visualize_prediction(model, img_path, class_names):
    """Визуализация предсказания с показанием."""
    reading, result = predict_meter_reading(model, img_path)

    # Отрисовка результата
    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(annotated_rgb)
    ax.set_title(f"Распознанное показание: {reading}", fontsize=16, fontweight="bold", color="green")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    return reading

In [ ]:
# Тестируем на нескольких изображениях из тестового набора
test_imgs = glob.glob(os.path.join(dataset.location, "test", "images", "*"))
np.random.seed(123)
samples = np.random.choice(test_imgs, min(8, len(test_imgs)), replace=False)

print("=== Тестирование на изображениях ===")
for img_path in samples:
    reading = visualize_prediction(best_model, img_path, class_names)
    print(f"  Файл: {os.path.basename(img_path)} → Показание: {reading}")

---
## 9. Экспорт модели для Android (TFLite)

In [ ]:
# Экспорт в TFLite (float16 — хороший баланс точность/размер)
tflite_model = best_model.export(
    format="tflite",
    imgsz=640,
    half=True  # float16 квантизация
)

print(f"\nМодель экспортирована: {tflite_model}")
print(f"Размер: {os.path.getsize(tflite_model) / 1024 / 1024:.2f} MB")

In [ ]:
# Также экспортируем INT8 квантизованную версию (меньше, быстрее на мобильных)
tflite_int8 = best_model.export(
    format="tflite",
    imgsz=640,
    int8=True
)

print(f"\nINT8 модель: {tflite_int8}")
print(f"Размер: {os.path.getsize(tflite_int8) / 1024 / 1024:.2f} MB")

---
## 10. Проверка TFLite модели

In [ ]:
# Загружаем и тестируем TFLite модель
tflite_model_loaded = YOLO(tflite_model)

# Тестируем на одном изображении
if test_imgs:
    test_img = test_imgs[0]
    print("=== Тест TFLite модели ===")
    reading_tflite, _ = predict_meter_reading(tflite_model_loaded, test_img)
    reading_pt, _ = predict_meter_reading(best_model, test_img)

    print(f"PyTorch модель → {reading_pt}")
    print(f"TFLite модель  → {reading_tflite}")
    print(f"Совпадение: {'✅' if reading_pt == reading_tflite else '❌'}")

---
## 11. Скачивание моделей

In [ ]:
# Скачиваем все артефакты
from google.colab import files
import shutil

# Создаём архив с моделями и результатами
export_dir = "meter_model_export"
os.makedirs(export_dir, exist_ok=True)

# Копируем best.pt
shutil.copy(best_model_path, os.path.join(export_dir, "best.pt"))

# Копируем TFLite модели
if os.path.exists(str(tflite_model)):
    shutil.copy(str(tflite_model), os.path.join(export_dir, "meter_reading_fp16.tflite"))

# Копируем графики
for plot_name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
                   "PR_curve.png", "F1_curve.png"]:
    src = os.path.join(results_dir, plot_name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(export_dir, plot_name))

# Архивируем
shutil.make_archive("meter_model_export", "zip", export_dir)
print("Архив создан: meter_model_export.zip")

# Скачиваем
files.download("meter_model_export.zip")

---
## 12. Сравнение: Transfer Learning vs Full Training

Для дипломной работы полезно показать сравнение подходов.

In [ ]:
# === Эксперимент: обучение без заморозки (Full Fine-tuning) ===
# Раскомментируйте для запуска (занимает больше времени)

# model_full = YOLO("yolov8n.pt")
# results_full = model_full.train(
#     data=data_yaml_path,
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     freeze=0,          # <-- НЕ замораживаем (обучаем всё)
#     patience=15,
#     optimizer="AdamW",
#     lr0=0.0005,        # <-- меньший lr для full fine-tuning
#     cos_lr=True,
#     device=0,
#     project="meter_reading",
#     name="yolov8n_full",
#     verbose=True,
#     plots=True
# )

In [ ]:
# === Эксперимент: обучение с нуля (from scratch) ===
# Раскомментируйте для запуска

# model_scratch = YOLO("yolov8n.yaml")  # <-- только архитектура, БЕЗ весов
# results_scratch = model_scratch.train(
#     data=data_yaml_path,
#     epochs=100,         # <-- больше эпох нужно для обучения с нуля
#     imgsz=640,
#     batch=16,
#     patience=20,
#     optimizer="AdamW",
#     lr0=0.01,
#     cos_lr=True,
#     device=0,
#     project="meter_reading",
#     name="yolov8n_scratch",
#     verbose=True,
#     plots=True
# )

### Ожидаемые результаты сравнения

| Подход | mAP@50 | Время обучения | Описание |
|---|---|---|---|
| Transfer Learning (freeze backbone) | ~95–97% | ~20 мин | Только голова обучена |
| Full Fine-tuning | ~96–98% | ~40 мин | Все слои дообучены |
| From Scratch | ~85–92% | ~80 мин | Без предобученных весов |

Transfer Learning даёт отличные результаты при минимальном времени обучения — **это главное преимущество подхода**.

---
## 13. Вывод

### Что получилось:
1. ✅ Модель YOLOv8n переобучена для распознавания цифр счётчиков
2. ✅ Transfer Learning: backbone заморожен, обучена только detection head
3. ✅ Датасет: 5.5k изображений, 10 классов (цифры 0–9)
4. ✅ Экспорт в TFLite для Android

### Использование в Android:
1. Положить файл `meter_reading_fp16.tflite` в `app/src/main/assets/`
2. Добавить зависимость `org.tensorflow:tensorflow-lite`
3. Заменить ML Kit на свой TFLite inference pipeline
4. Постобработка: сортировка bbox по x → конкатенация цифр → показание